In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
pip install unidecode python-calamine -q

In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import monotonically_increasing_id, split, trim, col, Column, coalesce, isnull, translate, round
from pyspark.sql.functions import avg, col, rand, when, lower, regexp_extract, regexp_replace, expr, create_map, lit
import pyspark.sql.functions as F
import pandas as pd
import geopandas as gpd
import unicodedata
from unidecode import unidecode
import matplotlib as plt
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import glob
from numpy import quantile
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import requests, io

In [18]:
spark = SparkSession.builder \
    .appName("Trata base de dados criminais") \
    .getOrCreate()

In [19]:
# path = "https://www.ssp.sp.gov.br/assets/estatistica/transparencia/spDados/SPDadosCriminais_2025.xlsx"

df_SPDadosCriminais = spark.createDataFrame(pd.concat([pd.read_excel(path, sheet_name="JAN_A_JUN_2025", engine="calamine"), 
                                                       pd.read_excel(path, sheet_name="JUL_A_DEZ_2025", engine="calamine")], 
                                                ignore_index=True).astype(str))


In [20]:
dados_categoria = [
    ("APREENSÃO DE ENTORPECENTES", "DROGAS"),
    ("ROUBO DE CARGA", "PATRIMONIAL"),
    ("HOMICÍDIO CULPOSO POR ACIDENTE DE TRÂNSITO", "TRÂNSITO"),
    ("HOMICÍDIO DOLOSO", "LETAL INTENCIONAL"),
    ("TENTATIVA DE HOMICÍDIO", "LETAL INTENCIONAL"),
    ("LATROCÍNIO", "LETAL INTENCIONAL"),
    ("LESÃO CORPORAL DOLOSA", "VIOLÊNCIA PESSOAL"),
    ("LESÃO CORPORAL CULPOSA - OUTRAS", "VIOLÊNCIA PESSOAL"),
    ("LESÃO CORPORAL SEGUIDA DE MORTE", "VIOLÊNCIA PESSOAL"),
    ("ROUBO - OUTROS", "PATRIMONIAL"),
    ("ROUBO DE VEÍCULO", "PATRIMONIAL"),
    ("FURTO - OUTROS", "PATRIMONIAL"),
    ("FURTO DE VEÍCULO", "PATRIMONIAL"),
    ("ESTUPRO TOTAL", "VIOLÊNCIA SEXUAL"),
    ("TRÁFICO DE ENTORPECENTES", "DROGAS"),
    ("PORTE DE ENTORPECENTES", "DROGAS"),
    ("PORTE DE ARMA", "ARMAS"),
    ("HOMICÍDIO CULPOSO OUTROS", "LETAL NÃO INTENCIONAL"),
    ("HOMICÍDIO DOLOSO POR ACIDENTE DE TRÂNSITO", "TRÂNSITO"),
    ("EXTORSÃO MEDIANTE SEQUESTRO", "VIOLÊNCIA GRAVE"),
    ("EXTORSÃO", "PATRIMONIAL"),
    ("SEQUESTRO E CÁRCERE PRIVADO", "VIOLÊNCIA GRAVE"),
    ("AMEAÇA", "VIOLÊNCIA PESSOAL"),
    ("VIAS DE FATO", "VIOLÊNCIA PESSOAL"),
    ("ESTELIONATO", "PATRIMONIAL"),
    ("RECEPTAÇÃO", "PATRIMONIAL"),
    ("DANO", "PATRIMONIAL"),
    ("INVASÃO DE DOMICÍLIO", "VIOLÊNCIA PESSOAL"),
    ("VIOLÊNCIA DOMÉSTICA", "VIOLÊNCIA PESSOAL"),
    ("ASSÉDIO SEXUAL", "VIOLÊNCIA SEXUAL"),
    ("IMPORTUNAÇÃO SEXUAL", "VIOLÊNCIA SEXUAL"),
    ("ATO OBSCENO", "VIOLÊNCIA SEXUAL"),
    ("CORRUPÇÃO DE MENORES", "VIOLÊNCIA SEXUAL"),
    ("MAUS TRATOS", "VIOLÊNCIA PESSOAL"),
    ("ABANDONO DE INCAPAZ", "VIOLÊNCIA PESSOAL"),
    ("OMISSÃO DE SOCORRO", "VIOLÊNCIA PESSOAL"),
    ("DESACATO", "OUTROS"),
    ("DESOBEDIÊNCIA", "OUTROS"),
    ("RESISTÊNCIA", "OUTROS"),
    ("USO DE DOCUMENTO FALSO", "OUTROS"),
    ("FALSIDADE IDEOLÓGICA", "OUTROS"),
    ("MOEDA FALSA", "OUTROS"),
    ("CRIME AMBIENTAL", "OUTROS"),
    ("CRIME ELEITORAL", "OUTROS"),
    ("CRIME MILITAR", "OUTROS"),
    ("OUTROS NÃO CRIMINAL", "OUTROS"),
    ("COMUNICAÇÃO DE ÓBITO", "OUTROS"),
    ("MORTE SUSPEITA", "OUTROS"),
    ("ENCONTRO DE CADÁVER", "OUTROS"),
    ("DESAPARECIMENTO DE PESSOA", "OUTROS"),
    ("LOCALIZAÇÃO DE VEÍCULO", "OUTROS"),
    ("LOCALIZAÇÃO DE PESSOA", "OUTROS"),
    ("APREENSÃO DE VEÍCULO", "OUTROS"),
    ("FURTO DE CARGA", "PATRIMONIAL"),
    ("LESÃO CORPORAL CULPOSA POR ACIDENTE DE TRÂNSITO", "TRÂNSITO"),
    ("RECUPERAÇÃO DE VEÍCULO", "OUTROS")]

df_categoria = spark.createDataFrame(dados_categoria, ["NATUREZA_APURADA", "Categoria_Crime"])

In [21]:
df_dados_criminais = (df_SPDadosCriminais.withColumn("LATITUDE",  F.trim(F.col("LATITUDE")))
                                         .withColumn("LONGITUDE", F.trim(F.col("LONGITUDE")))
                                         .withColumn("LATITUDE",  F.regexp_replace("LATITUDE",  ",", "."))
                                         .withColumn("LONGITUDE", F.regexp_replace("LONGITUDE", ",", "."))
                                         .withColumn("LATITUDE",  F.regexp_replace(F.col("LATITUDE"),  r"\.+$", ""))
                                         .withColumn("LONGITUDE", F.regexp_replace(F.col("LONGITUDE"), r"\.+$", ""))
                                         .filter(F.col("LATITUDE").isNotNull() &
                                                 F.col("LONGITUDE").isNotNull() &
                                                 (F.col("LATITUDE")  != "") &
                                                 (F.col("LONGITUDE") != "") &
                                                 (F.col("LATITUDE")  != "0") &
                                                 (F.col("LONGITUDE") != "0") &
                                                 (F.col("LATITUDE")  != "0.0") &
                                                 (F.col("LONGITUDE") != "0.0") &
                                                 (~F.upper(F.col("LATITUDE")).isin("NAN", "NULL", "NUL.L")) &
                                                 (~F.upper(F.col("LONGITUDE")).isin("NAN", "NULL", "NUL+L")))
                                         .withColumn("LATITUDE",  F.expr("try_cast(LATITUDE as double)"))
                                         .withColumn("LONGITUDE", F.expr("try_cast(LONGITUDE as double)"))
                                         .filter((col("COD IBGE") == "3550308") &  
                                                 F.col("LATITUDE").isNotNull() & 
                                                 F.col("LONGITUDE").isNotNull()  & 
                                                 F.col("LATITUDE").between(-24.0, -23.2) & 
                                                 F.col("LONGITUDE").between(-47.2, -46.2))
                                         .join(df_categoria, on="NATUREZA_APURADA", how="left"))


In [22]:
df_dados_criminais.write.mode("overwrite").parquet("drive/MyDrive/Colab Notebooks/Analise_Urbana_SP/BASES/dados_criminais//dados_criminais.parquet")